# Devan Experiments: Seed 2
Devan Rowland

**Running the notebook.**  This notebook trains all four models with training seed 2, using the team's shared code: VGG16, ResNet50 and MobileNetV2 with the two-stage recipe from Vivian's seed 0, and Joshua's custom CNN with his shared 50-epoch recipe.  It runs in Google Colab on a graphics processing unit (GPU), and it needs the same My Drive shortcut to Waste_Classification_Project as the evaluation notebook.  Since November 12 the shared code prepares every image with ImageNet normalization, and a check below confirms that before any training starts.  Each finished model's weight files are copied straight into a Seed_2 folder in the shared Drive folder, so nothing has to be downloaded or uploaded by hand.  A model whose final weight file is already in that folder is skipped, so a rerun only trains what's missing.  The same folder gets a training log with each model's best validation accuracy and training minutes, plus every epoch's loss and accuracy.  The three pretrained models took about 12 minutes together on a T4, and the custom CNN adds roughly 15 to 20 more, so keep the Colab tab open until the last cell finishes.  The seeds are fixed, but training on a GPU isn't exactly repeatable, so a rerun gives close but not identical numbers.  The setup comes first.

In [ ]:
import torch

# Check PyTorch and GPU
print(f"PyTorch {torch.__version__}")
gpu_available = torch.cuda.is_available()
print(f"GPU available: {gpu_available}")
if gpu_available:
    gpu_name = torch.cuda.get_device_name(0)
    print(gpu_name)

## Setup

**Shared code.**  The setup cell downloads the team's GitHub repository, or updates it if it's already there, and prints which commit it's using.  The commit should be dated November 15 or later, since that's when Joshua added the custom CNN's training function.  The same cell downloads the TrashNet images and unzips the resized set, and it should count 2,527 images.

**Seed.**  The seed cell fixes the random number generators in Python, NumPy and PyTorch to the training seed, 2.  The seed sets the starting weights of each new layer and the order the training images are shuffled in.  Changing that one number trains a different seed into its own Seed folder.

**Normalization check.**  The loader cell builds the shared training, validation and test loaders and prints the pixel range of one validation batch.  Normalized pixels run from about -2.1 to 2.6 instead of 0 to 1, so a negative minimum confirms the shared code's ImageNet normalization is on.

The training comes next.

In [ ]:
import glob
import os
import subprocess
import sys
import zipfile

# Set download locations
repo_url = (
    "https://github.com/viviantothewu-web/trashnet-waste-classification"
)
trashnet_url = "https://github.com/garythung/trashnet.git"
repo_folder = "/content/trashnet-waste-classification"
trashnet_folder = f"{repo_folder}/trashnet"
image_folder = f"{trashnet_folder}/data/dataset-resized"
image_zip_path = f"{trashnet_folder}/data/dataset-resized.zip"

# Clone shared repo
if not os.path.exists(repo_folder):
    repo_command = ["git", "clone", repo_url, repo_folder]
    subprocess.run(repo_command, check=True)

# Pull latest shared code
pull_command = ["git", "-C", repo_folder, "pull"]
subprocess.run(pull_command, check=True)

# Clone TrashNet images
if not os.path.exists(trashnet_folder):
    trashnet_command = ["git", "clone", trashnet_url, trashnet_folder]
    subprocess.run(trashnet_command, check=True)

# Unzip resized images
if not os.path.exists(image_folder):
    image_zip = zipfile.ZipFile(image_zip_path)
    image_zip.extractall(f"{trashnet_folder}/data")
    image_zip.close()

# Count TrashNet images
image_paths = glob.glob(f"{image_folder}/*/*.jpg")
image_total = len(image_paths)
print(f"TrashNet images: {image_total}")

# Move into shared repo
os.chdir(repo_folder)
sys.path.insert(0, repo_folder)

# Print shared code version
log_command = ["git", "log", "-1", "--date=short", "--format=%h %cd %s"]
log_result = subprocess.run(log_command, capture_output=True, text=True)
commit_line = log_result.stdout
commit_text = commit_line.strip()
print(f"Shared code commit: {commit_text}")

In [ ]:
import numpy as np
import random
import torch

# Set training seed
training_seed = 2
random.seed(training_seed)
np.random.seed(training_seed)
torch.manual_seed(training_seed)
torch.cuda.manual_seed_all(training_seed)
print(f"Training seed: {training_seed}")

In [ ]:
from src.data_pipeline import create_dataloaders

# Load shared split loaders
split_path = "data_splits/trashnet_split.csv"
train_loader, validation_loader, test_loader = create_dataloaders(
    split_path,
    img_size=224,
    batch_size=32,
    training_seed=training_seed,
)
train_batch_total = len(train_loader)
validation_batch_total = len(validation_loader)
print(f"Train batches: {train_batch_total}")
print(f"Validation batches: {validation_batch_total}")

# Check ImageNet normalization
validation_batches = iter(validation_loader)
check_images, check_labels = next(validation_batches)
lowest_value = float(check_images.min())
highest_value = float(check_images.max())
normalization_on = lowest_value < 0.0
print(f"Pixel range: {lowest_value:.2f} to {highest_value:.2f}")
print(f"ImageNet normalization on: {normalization_on}")

## Training

**Pretrained models.**  For VGG16, ResNet50 and MobileNetV2, the shared training function loads the ImageNet weights, swaps the 1,000-class output layer for a new 6-class one, and trains in two stages.  Stage 1 freezes everything except the new layer and trains it for 5 epochs with a learning rate of 0.001.  An epoch is one full pass through the 1,768 training images.  Stage 2 unfreezes the whole network and fine-tunes it for 5 more epochs at 0.0001, so the pretrained weights only shift a little.  Each stage keeps the weights from its best validation epoch.

**Custom CNN.**  The custom CNN starts from random weights, so it has no frozen stage.  Joshua's shared function trains it for 50 epochs with the Adam optimizer at a learning rate of 0.001 and keeps the weights from its best validation epoch.  Its log row puts that best validation accuracy in the stage 2 column, the column the evaluation notebook reads as each model's final number.

**Skipping.**  Before training a model, the run cell checks whether its final weight file is already in the Seed folder.  If it is, the model is skipped and its earlier log rows are kept, so a rerun after the pretrained models only trains the custom CNN.  Setting `retrain_existing` to `True` trains every model again.

**Saving.**  As soon as a model finishes, its weight files are copied to the Seed folder in the shared Drive, so a disconnect partway through loses at most the model in progress.  The run records each model's best validation accuracy, the epoch that gave it, the training minutes, the GPU and the shared code commit.  Those go to the training log, which the evaluation notebook reads for its training-log check.  Every epoch's numbers go to a history file for the report's training curves.

The results follow the run cell.

In [ ]:
import os
import pandas as pd
import shutil
import subprocess
import time
import torch
from src.custom_cnn_training import run_custom_cnn
from src.transfer_learning import run_transfer_learning

def pick_device():
    # Prefer GPU when present
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def describe_device(device):
    # Name GPU or CPU
    if device.type == "cuda":
        return torch.cuda.get_device_name(0)
    return "cpu"

def read_commit():
    # Read shared code commit
    log_command = ["git", "log", "-1", "--format=%h"]
    log_result = subprocess.run(log_command, capture_output=True, text=True)
    commit_line = log_result.stdout
    commit_text = commit_line.strip()
    return commit_text

def make_final_name(architecture, seed):
    # Name final weight file
    if architecture == "customcnn":
        return f"customcnn_seed{seed}.pth"
    return f"{architecture}_seed{seed}_stage2.pth"

def make_history_rows(architecture, seed, stage, history):
    # List epoch rows
    history_rows = []
    epoch_total = len(history["val_accuracy"])
    for epoch in range(epoch_total):
        epoch_number = epoch + 1
        history_row = {
            "architecture": architecture,
            "seed": seed,
            "stage": stage,
            "epoch": epoch_number,
            "train_loss": history["train_loss"][epoch],
            "train_accuracy": history["train_accuracy"][epoch],
            "val_loss": history["val_loss"][epoch],
            "val_accuracy": history["val_accuracy"][epoch],
        }
        history_rows.append(history_row)
    return history_rows

def train_and_save(architecture, seed, seed_folder, device):
    # Train both stages
    print(f"Training {architecture} seed {seed}")
    start_time = time.perf_counter()
    model, stage1_history, stage2_history = run_transfer_learning(
        architecture,
        train_loader,
        validation_loader,
        device,
        seed,
    )
    end_time = time.perf_counter()
    elapsed_seconds = end_time - start_time
    training_minutes = elapsed_seconds / 60

    # Free GPU memory
    del model
    torch.cuda.empty_cache()

    # Copy weights to Drive
    for stage in [1, 2]:
        file_name = f"{architecture}_seed{seed}_stage{stage}.pth"
        local_path = f"saved_models/{file_name}"
        drive_path = f"{seed_folder}/{file_name}"
        shutil.copy(local_path, drive_path)
        print(f"Saved {file_name} to Drive")

    # Find best validation epochs
    stage1_accuracies = stage1_history["val_accuracy"]
    stage2_accuracies = stage2_history["val_accuracy"]
    stage1_best = max(stage1_accuracies)
    stage2_best = max(stage2_accuracies)
    best_epoch_index = stage2_accuracies.index(stage2_best)
    best_epoch_number = best_epoch_index + 1
    best_percent = stage2_best * 100
    print(f"{architecture} seed {seed}: best validation {best_percent:.2f}%")

    # Build log and history
    device_label = describe_device(device)
    commit_text = read_commit()
    log_row = {
        "architecture": architecture,
        "seed": seed,
        "stage1_best_val": stage1_best,
        "stage2_best_val": stage2_best,
        "stage2_best_epoch": best_epoch_number,
        "training_minutes": training_minutes,
        "device": device_label,
        "shared_code_commit": commit_text,
    }
    stage1_rows = make_history_rows(architecture, seed, 1, stage1_history)
    stage2_rows = make_history_rows(architecture, seed, 2, stage2_history)
    history_rows = stage1_rows + stage2_rows
    return log_row, history_rows

def train_custom_and_save(seed, seed_folder, device):
    # Train custom CNN
    print(f"Training customcnn seed {seed}")
    start_time = time.perf_counter()
    model, history, checkpoint_path = run_custom_cnn(
        train_loader,
        validation_loader,
        device,
        seed,
    )
    end_time = time.perf_counter()
    elapsed_seconds = end_time - start_time
    training_minutes = elapsed_seconds / 60

    # Free GPU memory
    del model
    torch.cuda.empty_cache()

    # Copy weights to Drive
    file_name = f"customcnn_seed{seed}.pth"
    drive_path = f"{seed_folder}/{file_name}"
    shutil.copy(checkpoint_path, drive_path)
    print(f"Saved {file_name} to Drive")

    # Find best validation epoch
    accuracies = history["val_accuracy"]
    best_accuracy = max(accuracies)
    best_epoch_index = accuracies.index(best_accuracy)
    best_epoch_number = best_epoch_index + 1
    best_percent = best_accuracy * 100
    print(f"customcnn seed {seed}: best validation {best_percent:.2f}%")

    # Build log and history
    device_label = describe_device(device)
    commit_text = read_commit()
    log_row = {
        "architecture": "customcnn",
        "seed": seed,
        "stage1_best_val": None,
        "stage2_best_val": best_accuracy,
        "stage2_best_epoch": best_epoch_number,
        "training_minutes": training_minutes,
        "device": device_label,
        "shared_code_commit": commit_text,
    }
    history_rows = make_history_rows("customcnn", seed, 1, history)
    return log_row, history_rows

def train_any_and_save(architecture, seed, seed_folder, device):
    # Pick training recipe
    if architecture == "customcnn":
        return train_custom_and_save(seed, seed_folder, device)
    return train_and_save(architecture, seed, seed_folder, device)

def merge_saved_rows(table_path, new_table):
    # Keep rows not retrained
    if not os.path.exists(table_path):
        return new_table
    saved_table = pd.read_csv(table_path)
    if len(new_table) == 0:
        return saved_table
    new_names = new_table["architecture"]
    retrained_mask = saved_table["architecture"].isin(new_names)
    kept_rows = saved_table[~retrained_mask]
    merged_table = pd.concat([kept_rows, new_table])
    return merged_table

In [ ]:
import os
import pandas as pd
from google.colab import drive

# Connect to Drive
drive.mount("/content/drive")
project_folder = "/content/drive/MyDrive/Waste_Classification_Project"
seed_folder = f"{project_folder}/Seed_{training_seed}"

# Check shared folder shortcut
folder_found = os.path.exists(project_folder)
print(f"Shared Drive folder found: {folder_found}")
if folder_found:
    os.makedirs(seed_folder, exist_ok=True)

# Train missing models
retrain_existing = False
device = pick_device()
log_rows = []
history_rows = []
for architecture in ["vgg16", "resnet50", "mobilenetv2", "customcnn"]:
    final_name = make_final_name(architecture, training_seed)
    already_saved = os.path.exists(f"{seed_folder}/{final_name}")
    if already_saved and not retrain_existing:
        print(f"Already in Drive, skipping: {final_name}")
    if folder_found and (retrain_existing or not already_saved):
        log_row, model_history_rows = train_any_and_save(
            architecture,
            training_seed,
            seed_folder,
            device,
        )
        log_rows.append(log_row)
        history_rows.extend(model_history_rows)

# Save training log
log_path = f"{seed_folder}/seed{training_seed}_training_log.csv"
new_log = pd.DataFrame(log_rows)
merged_log = merge_saved_rows(log_path, new_log)
merged_log.to_csv(log_path, index=False)

# Save epoch history
history_path = f"{seed_folder}/seed{training_seed}_history.csv"
new_history = pd.DataFrame(history_rows)
merged_history = merge_saved_rows(history_path, new_history)
merged_history.to_csv(history_path, index=False)
print("Saved training log and history")

## Results

**Training log.**  The cell below reloads the training log from Drive and prints each model's best validation accuracy, the epoch it came from, and the training minutes.

**Validation curves.**  The first graph shows the pretrained models over their ten epochs.  Epochs 1 to 5 are stage 1 and epochs 6 to 10 are stage 2, so the jump where fine-tuning starts shows up at epoch 6.  The second graph shows the custom CNN over its 50 epochs, with training accuracy next to validation accuracy, so a widening gap between them would show overfitting.

**Next.**  Run the evaluation notebook again.  It finds the new weight files and this training log on its own.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import drive

# Load saved training log
drive.mount("/content/drive")
training_seed = 2
project_folder = "/content/drive/MyDrive/Waste_Classification_Project"
seed_folder = f"{project_folder}/Seed_{training_seed}"
log_table = pd.read_csv(f"{seed_folder}/seed{training_seed}_training_log.csv")
history_table = pd.read_csv(f"{seed_folder}/seed{training_seed}_history.csv")

# Print best accuracies
for log_row in log_table.itertuples():
    final_percent = log_row.stage2_best_val * 100
    epoch_text = f"best epoch {log_row.stage2_best_epoch}"
    minute_text = f"{log_row.training_minutes:.1f} minutes"
    final_text = f"best {final_percent:.2f}%, {epoch_text}, {minute_text}"
    print(f"{log_row.architecture}: {final_text}")

# Set curve labels
curve_labels = [
    ("vgg16", "VGG16"),
    ("resnet50", "ResNet50"),
    ("mobilenetv2", "MobileNetV2"),
]

# Plot pretrained accuracy
plt.figure(figsize=(8, 5))
for architecture, legend_label in curve_labels:
    architecture_mask = history_table["architecture"] == architecture
    architecture_rows = history_table[architecture_mask]
    accuracy_percent = architecture_rows["val_accuracy"] * 100
    epoch_count = len(architecture_rows)
    epochs = np.arange(1, epoch_count + 1)
    plt.plot(epochs, accuracy_percent, label=legend_label)
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.legend(loc="lower right")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import drive

# Load saved epoch history
drive.mount("/content/drive")
training_seed = 2
project_folder = "/content/drive/MyDrive/Waste_Classification_Project"
seed_folder = f"{project_folder}/Seed_{training_seed}"
history_table = pd.read_csv(f"{seed_folder}/seed{training_seed}_history.csv")

# Pick custom CNN rows
custom_mask = history_table["architecture"] == "customcnn"
custom_rows = history_table[custom_mask]
train_percent = custom_rows["train_accuracy"] * 100
validation_percent = custom_rows["val_accuracy"] * 100
epoch_count = len(custom_rows)
epochs = np.arange(1, epoch_count + 1)

# Plot custom CNN accuracy
plt.figure(figsize=(8, 5))
plt.plot(epochs, train_percent, label="Training")
plt.plot(epochs, validation_percent, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Custom CNN, seed 2")
plt.legend(loc="lower right")
plt.show()